# eSim Comparison 2026

This notebook provides a data-driven comparison of various eSim providers for a 144-day stay in Japan. It evaluates travel-specific eSims against long-term student options (like Campus SIM) to find the best balance between price, data volume, and network technology (4G/5G).

*Note: Prices are calculated based on exchange rates and plans available in early 2026. Costs may vary slightly based on your home country or current promotions.*

---

<div style="display: flex; flex-wrap: wrap; gap: 10px; align-items: center;">
    <img src="eSimProviders/airalo.svg" style="width: 12%; height: auto;">
    <img src="eSimProviders/holafly.svg" style="width: 12%; height: auto;">
    <img src="eSimProviders/roamless.svg" style="width: 12%; height: auto;">
    <img src="eSimProviders/saily.svg" style="width: 12%; height: auto;">
    <img src="eSimProviders/ubigi.svg" style="width: 12%; height: auto;">
    <img src="eSimProviders/campusSim.svg" style="width: 12%; height: auto;">
    <img src="eSimProviders/sakuraMobile.svg" style="width: 12%; height: auto;">
    <img src="eSimProviders/jetpac.svg" style="width: 12%; height: auto;">
</div>

---

## Setup

Import data and prepare it

In [4]:
import plotly.io as pio
# Setzt den Standard-Renderer auf HTML, damit die Daten im Notebook-File gespeichert werden
pio.renderers.default = "notebook_connected"

import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Load data from the CSV file
df = pd.read_csv('eSimProvider.csv')

# 2. Preparation: Sort by cost for a cleaner visual
df_sorted = df.sort_values("Cost_for_144_Days", ascending=True)

# 3. Preparation… Calculate price/GB
df_sorted['Price_per_GB'] = df_sorted['Price_EUR'] / df_sorted['Data_GB']

# 4. Preparation: Unlimited/Non-Unlimited Plans 
df_limited = df_sorted[df_sorted['Is_Unlimited'] == False].copy()
df_unlimited = df_sorted[df_sorted['Is_Unlimited'] == True].copy()

---

## Analyse

Create visuals and plot the charts

In [5]:
# 5. Create Main Visualization

# Bar plot – limited
fig_bar_limited = px.bar(
    df_limited, 
    x="Cost_for_144_Days",
    y="Provider",
    color="Data_GB",
    orientation='h',
    text_auto='.2f',
    hover_data=["Plan_Name", "Local_Carrier"],
    title="<b>Japan Connectivity: 144-Day Cost Comparison</b><br>Sorted by Total Budget (EUR) – Limited Data",
    labels={
        "Cost_for_144_Days": "Total Cost (€)", 
        "Network_Speed": "Network Technology",
        "Data_GB": "Data GB",
        "Plan_Name": "Plan Name",
        "Local_Carrier": "Local Carrier"
    },
    color_discrete_map={"5G": "#00edff", "4G/5G": "#7300ff"},
    template="plotly_dark"
)

# Styling
fig_bar_limited.update_layout(
    font = dict(family = "Verdana", size = 12),
    title_font=dict(size=22),
    xaxis_tickprefix = "€",
    yaxis = {'categoryorder':'total descending'}
)

# Bar plot – unlimited
fig_bar_unlimited = px.bar(
    df_unlimited, 
    x="Cost_for_144_Days",
    y="Provider",
    color="Price_EUR",
    orientation='h',
    text_auto='.2f',
    hover_data=["Plan_Name", "Local_Carrier"],
    title="<b>Japan Connectivity: 144-Day Cost Comparison</b><br>Sorted by Total Budget (EUR) – Unlimited Data",
    labels={
        "Cost_for_144_Days": "Total Cost (€)", 
        "Network_Speed": "Network Technology",
        "Price_EUR": "Price EUR",
        "Plan_Name": "Plan Name",
        "Local_Carrier": "Local Carrier"
    },
    color_discrete_map={"5G": "#00edff", "4G/5G": "#7300ff"},
    template="plotly_dark"
)

# Styling
fig_bar_unlimited.update_layout(
    font = dict(family = "Verdana", size = 12),
    title_font=dict(size=22),
    xaxis_tickprefix = "€",
    yaxis = {'categoryorder':'total descending'}
)

# Scatter Plot – unlimited
fig_scatter = px.scatter(
    df_limited, 
    x="Data_GB", 
    y="Price_per_GB", 
    size="Price_EUR", 
    color="Provider",
    text="Provider",
    title="<b>Data Efficiency Analysis</b><br>Cost per GB vs. Monthly Allowance",
    labels={"Price_per_GB": "Cost per 1 GB (€)", "Data_GB": "Data Volume per Month (GB)", "Price_EUR": "Price EUR"},
    template="plotly_dark"
)

# 1. Create a 1x2 Subplot (Left: Best Value, Right: Data Efficiency)
fig_final = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Top 5 Cheapest Plans (144 Days)", "Data Efficiency (Price per GB)"),
    column_widths=[0.6, 0.4]
)

# 2. Add Horizontal Bar Chart (Cheapest Plans)
top_5_cheap = df_limited.nsmallest(5, 'Cost_for_144_Days')
fig_final.add_trace(
    go.Bar(
        x=top_5_cheap['Cost_for_144_Days'],
        y=top_5_cheap['Provider'],
        orientation='h',
        name="Total Cost (€)",
        marker=dict(color='#00edff'),
        text=top_5_cheap['Cost_for_144_Days'].map('{:,.2f}€'.format),
        textposition='auto'
    ),
    row=1, col=1
)

# 3. Add Radar-style Scatter (Price per GB)
fig_final.add_trace(
    go.Scatter(
        x=df_limited['Data_GB'],
        y=df_limited['Price_per_GB'],
        mode='markers+text',
        name="Value Efficiency",
        text=df_limited['Provider'],
        textposition="top center",
        marker=dict(size=12, color='#7300ff', symbol='diamond')
    ),
    row=1, col=2
)

# 4. Final Styling
fig_final.update_layout(
    title_text="<b>Final Verdict: Japan eSim Strategy 2026</b>",
    template="plotly_dark",
    showlegend=False,
    font=dict(family="Verdana", size=11)
);

---

## Visuals

In [6]:
# 6. Plot the visuals

fig_bar_limited.show()
fig_bar_unlimited.show()

fig_scatter.update_traces(textposition='top center')
fig_scatter.show()

fig_final.update_xaxes(title_text="Total Budget (€)", row=1, col=1)
fig_final.update_xaxes(title_text="Included Data (GB)", row=1, col=2)
fig_final.update_yaxes(title_text="Cost per 1GB (€)", row=1, col=2)

fig_final.show()

---

## Final Recommendation and Strategy

Based on the 144-day analysis, here are the most efficient strategies for your stay in Japan:

<ol>
    <li>🥇 The "Student Choice": Campus SIM</li>
        <ul>
            <li><b>Best for</b>: Long-term residents (Study Abroad).</li>
            <li><b>Why</b>: It offers the lowest cost over 5 months while providing a real Japanese phone number. This is essential if you plan to open a bank account, use local delivery apps, or need to be reachable by the University.</li>
            <li><b>Network</b>: Runs on NTT Docomo (5G), providing the best coverage in rural areas and inside buildings.</li>
        </ul>
    <li>🥈 The "Performance Choice": Jetpac or Ubigi</li>
        <ul>
            <li><b>Best for</b>: High data users who move around cities frequently.</li>
            <li><b>Why</b>: These providers often offer multi-network switching (Docomo/Softbank/KDDI). If one network is congested in a crowded area like Shibuya, your phone can hop to another.</li>
            <li><b>Network</b>: Excellent 5G speeds in urban centers.</li>
        </ul>
    <li>🥉 The "Emergency Backup": Roamless or Airalo</li>
        <ul>
            <li><b>Best for</b>: The first few days or as a secondary SIM.</li>
            <li><b>Why</b>: These are instant to set up. Roamless is particularly cool because the credit never expires—you can load $5 and keep it as a safety net for the entire 144 days. In addition to that Airalo has a automatically renewal feature.</li>
        </ul>
</ol>

🚀 Implementation Checklist
<ul>
    <li><b>Compatibility</b>: Ensure your phone is Carrier Unlocked before leaving Austria.</li>
    <li><b>Timing</b>: For Campus SIM, start the email application 1 week before departure. You will need your Residence Card (which you get at the airport) to finalize activation.</li>
    <li><b>Data Saver</b>: Use University Wi-Fi for heavy downloads/updates to stay within your monthly GB limit.</li>
</ul>

---

*Project from Dominik Altmann | Exchange Student from Austria*